## 2.1 一阶马尔可夫模型 + 拉普拉斯平滑

已知：字符序列 "ababc"，词汇表 {'a','b','c'}，一阶马尔可夫模型 p(xt | xt-1)，加 1 平滑。

统计真实转移次数：
- a → b：2 次（位置 1→2，3→4）
- b → a：1 次（位置 2→3）
- b → c：1 次（位置 4→5）
- 其他转移：0 次

词汇表大小 |V| = 3

各前驱字符出现次数：
- count('a') = 2（位置 1, 3）
- count('b') = 2（位置 2, 4）

加 1 平滑公式：
p(xt | x_{t-1}) = (count(x_{t-1}, xt) + 1) / (count(x_{t-1}) + |V|)

1. p('a' | 'b') = (count('b','a') + 1) / (count('b') + 3) = (1 + 1) / (2 + 3) = 2/5 = 0.4

2. p('c' | 'b') = (count('b','c') + 1) / (count('b') + 3) = (1 + 1) / (2 + 3) = 2/5 = 0.4

答案：p('a'|'b') = 0.4，p('c'|'b') = 0.4


## 3.1 线性RNN梯度推导

定义：h_t = W_hh * h_{t-1} + W_hx * x_t，o_t = W_oh * h_t
损失：L = (1/2) * Σ_{t=1}^{T} (o_t - y_t)^2

对 W_hh 求梯度，通过时间反向传播（BPTT）：

∂L/∂W_hh = Σ_{t=1}^{T} (∂L/∂h_t) * (∂h_t/∂W_hh)

其中 ∂h_t/∂W_hh = h_{t-1} + W_hh * (∂h_{t-1}/∂W_hh)

展开到所有时间步：
∂L/∂W_hh = Σ_{t=1}^{T} Σ_{k=1}^{t} (∂L/∂o_t) * W_oh * (W_hh)^{t-k} * (∂h_k/∂W_hh)

简化为：
∂L/∂W_hh = Σ_{t=1}^{T} Σ_{k=1}^{t} (o_t - y_t) * W_oh * (W_hh)^{t-k} * h_{k-1}

梯度消失或爆炸的条件：
- 如果 |W_hh| < 1，则 (W_hh)^{t-k} 随 t-k 增大指数衰减 → 梯度消失
- 如果 |W_hh| > 1，则 (W_hh)^{t-k} 随 t-k 增大指数增长 → 梯度爆炸
- 如果 |W_hh| = 1，梯度稳定（但实际中非线性激活函数也会影响）


## 4.1 深度双向RNN参数数量

假设：
- L 层
- 每层隐藏单元数 H
- 输入维度 D
- 输出维度 O

双向RNN每层包含前向和后向两个RNN。

每层参数：
- 输入到隐藏（前向 + 后向）：2 * (D * H + H)  [权重 + 偏置]
- 隐藏到隐藏（前向 + 后向）：2 * (H * H + H)  [权重 + 偏置]

每层总参数 = 2DH + 2H + 2H^2 + 2H = 2DH + 2H^2 + 4H

L 层总计（不含输出层）：
总参数 = L * (2DH + 2H^2 + 4H)

输出层参数（如果包含）：
输出层输入维度为 2H（双向拼接），输出维度 O：
输出层参数 = 2H * O + O

包含输出层的完整模型参数：
总参数 = L * (2DH + 2H^2 + 4H) + 2H * O + O


## 5.1 Skip-gram 负采样损失函数

给定中心词 w_c，上下文词 w_o，负样本词 u_k（k=1..K，从噪声分布采样）。

负采样损失函数（对数似然，最大化正样本概率，最小化负样本概率）：

L = -log σ(v_c · u_o) - Σ_{k=1}^{K} log σ(-v_c · u_k)

其中 σ(x) = 1 / (1 + exp(-x)) 是 sigmoid 函数。

完整目标函数（最小化负对数似然）：
J = -[ log σ(v_c · u_o) + Σ_{k=1}^{K} log σ(-v_c · u_k) ]

从噪声分布 P_n(w) 中采样负样本：
- 常用噪声分布：Unigram 分布或其 3/4 次方（如 Word2Vec 中的做法）
- 采样方法：基于词频构建概率分布，然后随机采样 K 个词（不包含正样本）
- P_n(w) ∝ count(w)^{3/4} / Σ_v count(v)^{3/4}


## 6.1 缩放点积注意力计算

已知：Q ∈ R^{2×4}，K ∈ R^{3×4}，V ∈ R^{3×5}，d_k = 4

步骤1：计算得分矩阵 S = Q * K^T / sqrt(d_k) = Q * K^T / 2

S 的形状为 2×3，其中 S_{i,j} = (Q_i · K_j) / 2

步骤2：对 S 的每一行应用 softmax，得到注意力权重矩阵 A（形状 2×3）
A_{i,j} = exp(S_{i,j}) / Σ_{l=1}^{3} exp(S_{i,l})

步骤3：计算输出 O = A * V（形状 2×5）
O_i = Σ_{j=1}^{3} A_{i,j} * V_j

最终输出矩阵 O 的形状为 2×5。

In [ ]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数
    
    Args:
        text: 输入文本字符串
        n: 滑动窗口大小（特征序列长度）
    
    Returns:
        vocab: 词汇表字典 {词: ID}
        features: 特征列表,每个元素是长度为n的词列表
        labels: 标签列表，每个元素是对应的下一个词
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    # 按频率降序排序，频率相同按字母序
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    for i in range(len(words) - n):
        # 取当前窗口的n个词作为特征
        feature = words[i:i+n]
        # 下一个词作为标签
        label = words[i+n]
        features.append(feature)
        labels.append(label)
    
    return vocab, (features, labels)

# ============ 测试示例 ============
if __name__ == "__main__":
    # 测试输入
    test_text = "The time machine"
    n = 2
    
    vocab, (features, labels) = preprocess_text(test_text, n)
    
    print("词汇表:", vocab)
    print("特征列表:", features)
    print("标签列表:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征列表: [['the', 'time']]
标签列表: ['machine']


In [ ]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单元前向传播
    
    Args:
        x_t: 输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏权重，形状 (input_size, hidden_size)
        W_hh: 隐藏到隐藏权重，形状 (hidden_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)
    
    Returns:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播
    """
    # 计算隐藏状态：h_t = tanh(x_t @ W_hx + h_prev @ W_hh + b_h)
    # 注意：这里使用 @ 表示矩阵乘法
    a = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(a)
    
    # 缓存前向传播的中间值，用于反向传播
    cache = (x_t, h_prev, W_hx, W_hh, b_h, a, h_t)
    
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN单元反向传播
    
    Args:
        dh_next: 上游梯度,即损失对h_t的梯度,形状 (batch_size, hidden_size)
        cache: 前向传播缓存的元组 (x_t, h_prev, W_hx, W_hh, b_h, a, h_t)
    
    Returns:
        dx_t: 损失对x_t的梯度,形状 (batch_size, input_size)
        dh_prev: 损失对h_prev的梯度,形状 (batch_size, hidden_size)
        dW_hx: 损失对W_hx的梯度,形状 (input_size, hidden_size)
        dW_hh: 损失对W_hh的梯度,形状 (hidden_size, hidden_size)
        db_h: 损失对b_h的梯度,形状 (hidden_size,)
    """
    x_t, h_prev, W_hx, W_hh, b_h, a, h_t = cache
    batch_size, hidden_size = h_t.shape
    
    # tanh的导数：d(tanh(a))/da = 1 - tanh(a)^2 = 1 - h_t^2
    da = dh_next * (1 - h_t ** 2)  # 形状 (batch_size, hidden_size)
    
    # 计算各参数的梯度
    # dW_hx = x_t^T @ da
    dW_hx = x_t.T @ da  # 形状 (input_size, hidden_size)
    
    # dW_hh = h_prev^T @ da
    dW_hh = h_prev.T @ da  # 形状 (hidden_size, hidden_size)
    
    # db_h = sum(da, axis=0)
    db_h = np.sum(da, axis=0)  # 形状 (hidden_size,)
    
    # dx_t = da @ W_hx^T
    dx_t = da @ W_hx.T  # 形状 (batch_size, input_size)
    
    # dh_prev = da @ W_hh^T
    dh_prev = da @ W_hh.T  # 形状 (batch_size, hidden_size)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ============ 测试示例 ============
if __name__ == "__main__":
    # 设置随机种子
    np.random.seed(42)
    
    # 参数设置
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    # 随机生成输入和权重
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    # 前向传播
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("前向传播结果 - h_t 形状:", h_t.shape)
    print("h_t:\n", h_t)
    
    # 反向传播
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)
    
    print("\n反向传播梯度形状:")
    print("dx_t:", dx_t.shape)
    print("dh_prev:", dh_prev.shape)
    print("dW_hx:", dW_hx.shape)
    print("dW_hh:", dW_hh.shape)
    print("db_h:", db_h.shape)

前向传播结果 - h_t 形状: (2, 4)
h_t:
 [[-0.99954071  0.88247634 -0.99663661 -0.61695335]
 [ 0.76489073 -0.97583963 -0.99964241 -0.370152  ]]

反向传播梯度形状:
dx_t: (2, 3)
dh_prev: (2, 4)
dW_hx: (3, 4)
dW_hh: (4, 4)
db_h: (4,)


In [4]:
import numpy as np

def bidirectional_rnn_encoder(X, W_hx_f, W_hh_f, b_h_f, W_hx_b, W_hh_b, b_h_b):
    """
    双向RNN编码器
    
    Args:
        X: 输入序列，形状 (seq_len, batch, input_dim)
        W_hx_f: 前向输入到隐藏权重，(input_dim, hidden_dim)
        W_hh_f: 前向隐藏到隐藏权重，(hidden_dim, hidden_dim)
        b_h_f: 前向偏置，(hidden_dim,)
        W_hx_b: 后向输入到隐藏权重，(input_dim, hidden_dim)
        W_hh_b: 后向隐藏到隐藏权重，(hidden_dim, hidden_dim)
        b_h_b: 后向偏置，(hidden_dim,)
    
    Returns:
        outputs: 每个时间步拼接的前后向隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
        final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
    """
    seq_len, batch, input_dim = X.shape
    hidden_dim = W_hh_f.shape[0]
    
    # 前向传播
    h_f = np.zeros((batch, hidden_dim))  # 初始隐藏状态为0
    h_f_list = []
    
    for t in range(seq_len):
        x_t = X[t]  # (batch, input_dim)
        # h_t = tanh(x_t @ W_hx + h_prev @ W_hh + b_h)
        a_f = x_t @ W_hx_f + h_f @ W_hh_f + b_h_f
        h_f = np.tanh(a_f)
        h_f_list.append(h_f)
    
    # 后向传播（从后往前）
    h_b = np.zeros((batch, hidden_dim))  # 初始隐藏状态为0
    h_b_list = []
    
    for t in range(seq_len - 1, -1, -1):
        x_t = X[t]  # (batch, input_dim)
        a_b = x_t @ W_hx_b + h_b @ W_hh_b + b_h_b
        h_b = np.tanh(a_b)
        h_b_list.insert(0, h_b)  # 插入到开头，保持时间顺序
    
    # 拼接每个时间步的前向和后向隐藏状态
    outputs = []
    for t in range(seq_len):
        h_concat = np.concatenate([h_f_list[t], h_b_list[t]], axis=-1)  # (batch, 2*hidden_dim)
        outputs.append(h_concat)
    
    outputs = np.stack(outputs, axis=0)  # (seq_len, batch, 2*hidden_dim)
    
    # 最终时间步的拼接隐藏状态（最后一个时间步的前向 + 第一个时间步的后向）
    final_state = np.concatenate([h_f_list[-1], h_b_list[0]], axis=-1)  # (batch, 2*hidden_dim)
    
    return outputs, final_state


# ============ 使用 PyTorch 的实现（更符合作业要求） ============
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    """使用PyTorch实现的双向RNN编码器"""
    
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BiRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 使用PyTorch的RNN，设置 bidirectional=True
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=False,  # 使用 (seq, batch, feature) 格式
            bidirectional=True,
            nonlinearity='tanh'
        )
    
    def forward(self, X):
        """
        Args:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        Returns:
            outputs: 形状 (seq_len, batch, 2*hidden_dim)
            final_state: 形状 (batch, 2*hidden_dim)
        """
        # 前向传播
        outputs, h_n = self.rnn(X)
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (num_layers * 2, batch, hidden_dim)
        
        # 获取最终时间步的拼接隐藏状态
        # 前向最后一个: h_n[-2, :, :]，后向最后一个: h_n[-1, :, :]
        final_forward = h_n[-2, :, :]  # (batch, hidden_dim)
        final_backward = h_n[-1, :, :]  # (batch, hidden_dim)
        final_state = torch.cat([final_forward, final_backward], dim=-1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state


# ============ 测试示例 ============
if __name__ == "__main__":
    # 参数设置
    seq_len = 5
    batch = 3
    input_dim = 4
    hidden_dim = 6
    
    # 随机生成输入
    X = np.random.randn(seq_len, batch, input_dim)
    X_torch = torch.tensor(X, dtype=torch.float32)
    
    # 使用PyTorch实现
    model = BiRNNEncoder(input_dim, hidden_dim)
    outputs, final_state = model(X_torch)
    
    print("使用PyTorch实现:")
    print("outputs 形状:", outputs.shape)  # 预期: (5, 3, 12)
    print("final_state 形状:", final_state.shape)  # 预期: (3, 12)
    
    # 手动实现验证（使用随机权重）
    print("\n手动实现:")
    W_hx_f = np.random.randn(input_dim, hidden_dim)
    W_hh_f = np.random.randn(hidden_dim, hidden_dim)
    b_h_f = np.random.randn(hidden_dim)
    W_hx_b = np.random.randn(input_dim, hidden_dim)
    W_hh_b = np.random.randn(hidden_dim, hidden_dim)
    b_h_b = np.random.randn(hidden_dim)
    
    outputs_manual, final_state_manual = bidirectional_rnn_encoder(
        X, W_hx_f, W_hh_f, b_h_f, W_hx_b, W_hh_b, b_h_b
    )
    
    print("outputs_manual 形状:", outputs_manual.shape)  # 预期: (5, 3, 12)
    print("final_state_manual 形状:", final_state_manual.shape)  # 预期: (3, 12)

使用PyTorch实现:
outputs 形状: torch.Size([5, 3, 12])
final_state 形状: torch.Size([3, 12])

手动实现:
outputs_manual 形状: (5, 3, 12)
final_state_manual 形状: (3, 12)


In [9]:
import numpy as np

def cbow_forward(context_indices, W, W_out, context_size):
    """
    CBOW模型前向传播
    
    参数:
        context_indices: 上下文词索引，形状 (batch_size, context_size)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
        context_size: 上下文词数量
    
    返回:
        probs: 输出概率分布，形状 (batch_size, V)
    """
    batch_size, ctx_size = context_indices.shape
    
    # 检查上下文大小是否匹配
    assert ctx_size == context_size, f"期望 context_size={context_size}，实际得到 {ctx_size}"
    
    # 1. 获取每个上下文词的词向量并求平均
    word_vectors = W[context_indices]  # (batch_size, context_size, d)
    h = np.mean(word_vectors, axis=1)  # (batch_size, d) - 平均上下文向量
    
    # 2. 计算输出分数和概率分布（完整 Softmax）
    scores = h @ W_out  # (batch_size, V)
    
    # Softmax：exp(x) / sum(exp(x))，减去最大值防止数值溢出
    exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)  # (batch_size, V)
    
    return probs


def cbow_loss(context_indices, target_indices, W, W_out, context_size):
    """
    CBOW模型完整前向传播 + 交叉熵损失计算
    
    参数:
        context_indices: 上下文词索引，形状 (batch_size, context_size)
        target_indices: 目标中心词索引，形状 (batch_size,)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
        context_size: 上下文词数量
    
    返回:
        loss: 交叉熵损失（标量）
        probs: 输出概率分布，形状 (batch_size, V)
    """
    batch_size = context_indices.shape[0]
    
    # 前向传播得到概率分布
    probs = cbow_forward(context_indices, W, W_out, context_size)
    
    # 计算交叉熵损失
    # L = - (1/batch_size) * sum(log(p(target_i)))
    batch_indices = np.arange(batch_size)
    target_probs = probs[batch_indices, target_indices]  # 每个样本目标词的概率
    
    # 添加极小常数防止 log(0)
    eps = 1e-8
    loss = -np.mean(np.log(target_probs + eps))
    
    return loss, probs


# ============ 测试 ============
if __name__ == "__main__":
    # 参数设置
    V = 10          # 词汇表大小
    d = 4           # 嵌入维度
    context_size = 3  # 上下文词数量
    batch_size = 2   # 批次大小
    
    # 随机生成权重
    np.random.seed(42)
    W = np.random.randn(V, d)           # 输入权重 (V, d)
    W_out = np.random.randn(d, V)       # 输出权重 (d, V)
    
    # 随机生成一批样本
    # context_indices: (batch_size, context_size)，每个样本有3个上下文词
    context_indices = np.random.randint(0, V, size=(batch_size, context_size))
    # target_indices: (batch_size,)，每个样本的目标中心词（一维数组）
    target_indices = np.random.randint(0, V, size=batch_size)
    
    print("上下文词索引:")
    print(context_indices)
    print("目标词索引:", target_indices)
    print("W 形状:", W.shape)
    print("W_out 形状:", W_out.shape)
    
    # 计算前向传播和损失
    probs = cbow_forward(context_indices, W, W_out, context_size)
    loss, probs_with_loss = cbow_loss(context_indices, target_indices, W, W_out, context_size)
    
    print("\n输出概率分布形状:", probs.shape)
    print("概率分布:")
    print(probs)
    print("每行概率和:", np.sum(probs, axis=1))
    print("交叉熵损失:", loss)

上下文词索引:
[[4 6 6]
 [3 6 2]]
目标词索引: [5 1]
W 形状: (10, 4)
W_out 形状: (4, 10)

输出概率分布形状: (2, 10)
概率分布:
[[0.03759965 0.02777842 0.12212262 0.13547509 0.09761387 0.01475736
  0.04861344 0.00635717 0.02142616 0.48825622]
 [0.07214654 0.06279349 0.2998514  0.1428666  0.04306387 0.00954282
  0.10800431 0.01921774 0.03311524 0.209398  ]]
每行概率和: [1. 1.]
交叉熵损失: 3.4919582549093002


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """多头注意力机制"""
    
    def __init__(self, d_model, num_heads):
        """
        Args:
            d_model: 模型维度
            num_heads: 注意力头数
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # 最终输出线性层
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, X):
        """
        Args:
            X: 输入，形状 (seq_len, batch, d_model)
        
        Returns:
            output: 形状 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 1. 线性投影得到 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)
        
        # 2. 重塑为多头格式: (seq_len, batch, num_heads, d_k) 
        #    -> 转置为 (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(0, 1)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(0, 1)
        V = V.view(seq_len, batch, self.num_heads, self.d_v).transpose(0, 2).transpose(0, 1)
        # Q, K, V 形状: (batch, num_heads, seq_len, d_k)
        
        # 3. 缩放点积注意力
        # 计算得分: Q @ K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        # scores 形状: (batch, num_heads, seq_len, seq_len)
        
        # Softmax
        attn_weights = F.softmax(scores, dim=-1)
        
        # 加权求和: attn_weights @ V
        attn_output = torch.matmul(attn_weights, V)
        # attn_output 形状: (batch, num_heads, seq_len, d_v)
        
        # 4. 拼接所有头
        # 转置回 (seq_len, batch, num_heads, d_v) -> 重塑为 (seq_len, batch, d_model)
        attn_output = attn_output.transpose(0, 2).transpose(0, 1).contiguous()
        # 现在形状: (seq_len, batch, num_heads, d_v)
        attn_output = attn_output.view(seq_len, batch, self.d_model)
        
        # 5. 最终线性层
        output = self.W_o(attn_output)
        
        return output


# ============ 测试示例 ============
if __name__ == "__main__":
    # 参数设置
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch = 2
    
    # 随机生成输入
    X = torch.randn(seq_len, batch, d_model)
    
    # 创建多头注意力模型
    model = MultiHeadAttention(d_model, num_heads)
    
    # 前向传播
    output = model(X)
    
    print("输入 X 形状:", X.shape)  # (3, 2, 4)
    print("输出 output 形状:", output.shape)  # (3, 2, 4)
    print("\n是否与输入形状相同:", X.shape == output.shape)
    
    # 验证每个头的维度
    print(f"\n配置: d_model={d_model}, num_heads={num_heads}")
    print(f"每个头的维度 d_k = d_v = {d_model // num_heads}")

输入 X 形状: torch.Size([3, 2, 4])
输出 output 形状: torch.Size([3, 2, 4])

是否与输入形状相同: True

配置: d_model=4, num_heads=2
每个头的维度 d_k = d_v = 2
